# Benchmarks for Point Source to Far Field Reflector Problem

This notebook runs the benchmark cases for the Point Source to Far Field reflector problem using Sinkhorn divergence-based optimal transport.

**Available benchmark cases:**
- `3D_SquareToCircle_logcost_MonteCarlo` — uniform square source → uniform circle target
- `3D_SquareToTwoGaussSide_logcost_MonteCarlo` — uniform square source → two-Gaussian mixture target

**Requirements:** Intel OneAPI MKL must be installed. The notebook will auto-detect MKL paths and patch the Makefile. If MKL is not found, follow the Docker instructions in the README.

## 1. Configuration

Choose the benchmark test case.

In [ ]:
import os
import re
import glob
import subprocess
import numpy as np

# Choose benchmark: 'SquareToCircle' or 'SquareToTwoGaussSide'
BENCHMARK = 'SquareToCircle'

BENCHMARK_HEADERS = {
    'SquareToCircle':       'Benchmarks/test_3D_SquareToCircle_logcost_MonteCarlo.h',
    'SquareToTwoGaussSide': 'Benchmarks/test_3D_SquareToTwoGaussSide_logcost_MonteCarlo.h',
}

assert BENCHMARK in BENCHMARK_HEADERS, (
    f"Unknown benchmark '{BENCHMARK}'. Choose from: {list(BENCHMARK_HEADERS)}"
)

chosen_header = BENCHMARK_HEADERS[BENCHMARK]
print(f"Selected benchmark : {BENCHMARK}")
print(f"Header file        : {chosen_header}")

## 2. Patch main.cpp to Select the Benchmark

The first `#include` in `main.cpp` controls which test case is active. This cell comments out the inactive case and uncomments the chosen one.

In [ ]:
main_cpp_path = os.path.join('BenchmarkCode', 'main.cpp')

with open(main_cpp_path, 'r') as fh:
    src = fh.read()

for h in BENCHMARK_HEADERS.values():
    if h == chosen_header:
        # Uncomment if currently commented
        src = re.sub(
            r'^//\s*(#include\s+"' + re.escape(h) + r'")',
            r'\1', src, flags=re.MULTILINE
        )
    else:
        # Comment out if currently active
        src = re.sub(
            r'^(#include\s+"' + re.escape(h) + r'")',
            r'//\1', src, flags=re.MULTILINE
        )

with open(main_cpp_path, 'w') as fh:
    fh.write(src)

active = [l.strip() for l in src.splitlines()
          if l.strip().startswith('#include') and 'Benchmarks/' in l]
print("Active benchmark includes:")
for inc in active:
    print(" ", inc)

## 3. Detect MKL and Patch the Makefile

The Makefile was written for the Docker image (`/opt/intel/oneapi/mkl/latest/`). This cell searches for MKL on the current machine and rewrites the Makefile with the correct paths.

In [ ]:
def find_file(filename, search_roots):
    """Return the first path that contains `filename` under any of `search_roots`."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        result = subprocess.run(
            ['find', root, '-name', filename, '-print', '-quit'],
            capture_output=True, text=True
        )
        hit = result.stdout.strip()
        if hit:
            return hit
    return None

SEARCH_ROOTS = [
    '/opt/intel',
    '/usr/local/intel',
    '/usr',
    os.path.expanduser('~/intel'),
]

mkl_h    = find_file('mkl.h',                    SEARCH_ROOTS)
mkl_core = find_file('libmkl_core.a',            SEARCH_ROOTS)

if not mkl_h:
    raise FileNotFoundError(
        "Cannot find mkl.h. Install Intel OneAPI MKL or run via Docker:\n"
        "  docker build -t benchmarks-ps-reflector .\n"
        "  docker run -it --rm -p 8888:8888 benchmarks-ps-reflector "
        "jupyter-lab --ip=0.0.0.0 --port=8888 --allow-root"
    )
if not mkl_core:
    raise FileNotFoundError(
        "Found mkl.h but cannot find libmkl_core.a. "
        "Check your MKL installation."
    )

mkl_include_dir = os.path.dirname(mkl_h)
mkl_lib_dir     = os.path.dirname(mkl_core)

print(f"MKL include : {mkl_include_dir}")
print(f"MKL lib     : {mkl_lib_dir}")

# ---- Patch the Makefile ----
makefile_path = os.path.join('BenchmarkCode', 'Makefile')

with open(makefile_path, 'r') as fh:
    mk = fh.read()

# Replace -I <old include path>
mk = re.sub(r'-I\s+\S+/include(?=\s)', f'-I {mkl_include_dir}', mk)

# Replace every occurrence of the old lib dir in the static-lib paths
mk = re.sub(r'\S+/lib/intel64/', f'{mkl_lib_dir}/', mk)

with open(makefile_path, 'w') as fh:
    fh.write(mk)

print("\nPatched Makefile:")
print(mk)

## 4. Compile

In [ ]:
%cd BenchmarkCode
!make 2>&1

## 5. Run the Benchmark

Timing is printed to the terminal; all data files are saved into a timestamped `Results_{testname}/` subfolder.

In [ ]:
!./main 2>&1
print("\nBenchmark complete.")

## 6. Locate and Read Output Files

In [ ]:
testname_map = {
    'SquareToCircle':       '3D_SquareToCircle_logcost_MonteCarlo',
    'SquareToTwoGaussSide': '3D_SquareToTwoGaussSide_logcost_MonteCarlo',
}
testname = testname_map[BENCHMARK]

results_root = f'Results_{testname}'
output_dirs = sorted(glob.glob(os.path.join(results_root, '*')))
assert output_dirs, (
    f"No output found under '{results_root}'. Did the benchmark run successfully?"
)

output_dir = output_dirs[-1]   # most recent run
print(f"Reading results from: {output_dir}")
print("Files:", [os.path.basename(f) for f in sorted(glob.glob(os.path.join(output_dir, '*.txt')))])


def read_vector(path):
    """Read a file written by print(double[], size): first line is count, rest are values."""
    with open(path) as fh:
        lines = [l.strip() for l in fh if l.strip()]
    n = int(lines[0])
    return np.array([float(l) for l in lines[1:n + 1]])


def read_matrix(path, ncols):
    """Read a file written by print(double[][dim], size1, dim): first line is row count."""
    with open(path) as fh:
        lines = [l.strip() for l in fh if l.strip()]
    n = int(lines[0])
    return np.array([[float(v) for v in l.split()] for l in lines[1:n + 1]])

In [ ]:
dim = 3

x    = read_matrix(os.path.join(output_dir, 'x_MY.txt'),    dim)
y    = read_matrix(os.path.join(output_dir, 'y_MY.txt'),    dim)
Ref  = read_matrix(os.path.join(output_dir, 'Ref_MY.txt'),  dim)
Refc = read_matrix(os.path.join(output_dir, 'Refc_MY.txt'), dim)
R    = read_vector(os.path.join(output_dir, 'R_MY.txt'))
p    = read_vector(os.path.join(output_dir, 'p_MY.txt'))
q    = read_vector(os.path.join(output_dir, 'q_MY.txt'))
f    = read_vector(os.path.join(output_dir, 'f_MY.txt'))
g    = read_vector(os.path.join(output_dir, 'g_MY.txt'))
f_id = read_vector(os.path.join(output_dir, 'f_id_MY.txt'))
g_id = read_vector(os.path.join(output_dir, 'g_id_MY.txt'))
fc   = read_vector(os.path.join(output_dir, 'fc_MY.txt'))
gc   = read_vector(os.path.join(output_dir, 'gc_MY.txt'))

print(f"NK              = {len(p)}")
print(f"Source points x : {x.shape}")
print(f"Target points y : {y.shape}")
print(f"Reflector Ref   : {Ref.shape}")

log_path = os.path.join(output_dir, 'log.txt')
with open(log_path) as fh:
    log_text = fh.read()
print("\n--- log.txt ---")
print(log_text)

## 7. Visualisation

### 7.1 Source and Target Distributions (stereographic projections)

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive; change to 'TkAgg' for local popup windows
import matplotlib.pyplot as plt

def stereo_north(pts):  # upper hemisphere -> plane via south-pole projection
    return pts[:, 0] / (1 + pts[:, 2]), pts[:, 1] / (1 + pts[:, 2])

def stereo_south(pts):  # lower hemisphere -> plane via north-pole projection
    mask = pts[:, 2] < 1.0
    denom = np.where(mask, 1 - pts[:, 2], np.nan)
    return pts[:, 0] / denom, pts[:, 1] / denom

xp1, xp2 = stereo_north(x)
yp1, yp2 = stereo_south(y)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc = axes[0].scatter(xp1, xp2, c=p, s=2, cmap='viridis')
plt.colorbar(sc, ax=axes[0], label='p (source density)')
axes[0].set_title('Source P  (north-pole projection)')
axes[0].set_xlabel('u'); axes[0].set_ylabel('v')
axes[0].set_aspect('equal')

sc = axes[1].scatter(yp1, yp2, c=q, s=2, cmap='plasma')
plt.colorbar(sc, ax=axes[1], label='q (target density)')
axes[1].set_title('Target Q  (south-pole projection)')
axes[1].set_xlabel('u'); axes[1].set_ylabel('v')
axes[1].set_aspect('equal')

plt.suptitle(f'Benchmark: {BENCHMARK}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'distributions.png'), dpi=120)
plt.show()
print("Saved distributions.png")

### 7.2 Reflector Surface (3-D scatter)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
sc = ax1.scatter(Ref[:, 0], Ref[:, 1], Ref[:, 2], c=R, s=2, cmap='coolwarm')
plt.colorbar(sc, ax=ax1, label='R (reflector radius)', shrink=0.6)
ax1.set_title('Reflector  (Sinkhorn divergence)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(122, projection='3d')
sc2 = ax2.scatter(Refc[:, 0], Refc[:, 1], Refc[:, 2],
                  c=np.exp(gc), s=2, cmap='coolwarm')
plt.colorbar(sc2, ax=ax2, label='exp(gc)', shrink=0.6)
ax2.set_title('Reflector  (c-transform)')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.suptitle(f'Reflector surfaces — {BENCHMARK}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'reflector_3d.png'), dpi=120)
plt.show()
print("Saved reflector_3d.png")

### 7.3 Kantorovich Potentials and Entropic Bias Correction

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

def scatter2d(ax, u, v, c, title, clabel):
    sc = ax.scatter(u, v, c=c, s=2, cmap='RdBu_r')
    plt.colorbar(sc, ax=ax, label=clabel)
    ax.set_title(title)
    ax.set_aspect('equal')

scatter2d(axes[0, 0], xp1, xp2, f,         'f  (source potential)',   'f')
scatter2d(axes[0, 1], xp1, xp2, f_id,      'f_id  (identity bias)',   'f_id')
scatter2d(axes[0, 2], xp1, xp2, f - f_id,  'f - f_id  (corrected)',   'f-f_id')

scatter2d(axes[1, 0], yp1, yp2, g,         'g  (target potential)',   'g')
scatter2d(axes[1, 1], yp1, yp2, g_id,      'g_id  (identity bias)',   'g_id')
scatter2d(axes[1, 2], yp1, yp2, g - g_id,  'g - g_id  (corrected)',   'g-g_id')

plt.suptitle(f'Kantorovich potentials & entropic bias — {BENCHMARK}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'potentials.png'), dpi=120)
plt.show()
print("Saved potentials.png")

### 7.4 C-transform Consistency Check

In [ ]:
dif_f = read_vector(os.path.join(output_dir, 'dif_fc_MY.txt'))
dif_g = read_vector(os.path.join(output_dir, 'dif_gc_MY.txt'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dif_f[np.isfinite(dif_f)], bins=60, color='steelblue', edgecolor='none')
axes[0].set_title('f - gc  (≈ 0 at convergence)')
axes[0].set_xlabel('f[i] - gc[i]')

axes[1].hist(dif_g[np.isfinite(dif_g)], bins=60, color='tomato', edgecolor='none')
axes[1].set_title('g - fc  (≈ 0 at convergence)')
axes[1].set_xlabel('g[j] - fc[j]')

plt.suptitle(f'C-transform residuals — {BENCHMARK}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'ctransform_residuals.png'), dpi=120)
plt.show()

print(f"  max |f - gc| = {np.nanmax(np.abs(dif_f)):.3e}")
print(f"  max |g - fc| = {np.nanmax(np.abs(dif_g)):.3e}")

## 8. Summary Statistics

In [ ]:
mask_p = p > 0
mask_q = q > 0
cost = (np.dot(f[mask_p] - f_id[mask_p], p[mask_p]) +
        np.dot(g[mask_q] - g_id[mask_q], q[mask_q]))

print("=" * 50)
print(f"Benchmark          : {BENCHMARK}")
print(f"NK (grid size)     : {len(p)}")
print(f"Sinkhorn div. cost : {cost:.6e}")
print(f"Reflector R range  : [{R.min():.4f}, {R.max():.4f}]")
print(f"Output directory   : {output_dir}")
print("=" * 50)